In [ ]:
# Install dependencies
# CC-MMD 2026 — Improved notebook v2
# Changes over v1:
#   1. Fixed is_cross_cultural key typo (was is_cross_culture) — cross-cultural prompt now fires
#   2. Stronger cross-cultural prompt with explicit Indian/Chinese cultural markers
#   3. Chain-of-thought reasoning in few-shot example answers
#   4. Fallback recovery added to batch path (was single-path only)
#   5. GPU-adaptive settings: T4 uses 4-bit+sdpa, A100 uses bfloat16+flash_attention_2
#   6. score_partition fixed (accuracy bug + norm_label inline)
#   7. DEBUG_LIMIT config flag for quick validation runs

!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q 'qwen-vl-utils[decord]==0.0.8'
!pip install -q pandas tqdm Pillow scikit-learn bitsandbytes



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR   = "/content/drive/MyDrive/data/cc_mmd_dataset"
MAMI_DIR   = "/content/drive/MyDrive/data/MAMI"
MDMD_DIR = "/content/drive/MyDrive/data/tamil"
MDMD_MALAYALAM_DIR = "/content/drive/MyDrive/data/malayalam"
CMMD_DIR = "/content/drive/MyDrive/data/CMMD"
OUTPUT_DIR = "/content/results"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
print("HF token:", "set" if os.environ["HF_TOKEN"] else "NOT SET")


In [ ]:
# ── CONFIG — edit before running ─────────────────────────────────────────────
import os

SELECTED_MODELS = [
    "Qwen/Qwen2.5-VL-7B-Instruct",
]

RUN_PARTITIONS = [
    # ── Malayalam: the three-way comparison for Table 7.2 ────────────────────
    "mdmd_original_malayalam",              # ZERO-SHOT      (no exemplars)
    "mdmd_original_malayalam_few_shot_tamil",  # FEW-SHOT, Tamil/MDMD exemplars
                                            #   -> lens-matched config; comparable
                                            #      with Tamil + CMMD rows. This is the
                                            #      n=200 redo of the old 0.6139 (n=198).
    "mdmd_original_malayalam_few_shot",     # FEW-SHOT, Malayalam exemplars
                                            #   -> optional lens-vs-domain confound test

    # ── Optional: Tamil redo at n=356 (old run was n=353 via the ID-collision bug)
    #"mdmd_original",

    # ── Unaffected by the exclusion bug; do NOT rerun (already n=340 / n=1000)
    #"cmmd_original",
    #"mami_indian",
    #"mami_chinese",
]

BATCH_SIZE  = 2     # Use 1 to enable per-image fallback on JSON failures
DEBUG_LIMIT = None  # Set to e.g. 10 for quick test; None for full run

print(f"Models     : {SELECTED_MODELS}")
print(f"Partitions : {RUN_PARTITIONS}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Debug limit: {DEBUG_LIMIT}")


In [ ]:
# ── Few-shot examples (partition-specific, balanced 5M:5NM each) ─────────────
# INDIAN_FEW_SHOT: drawn exclusively from MDMD dev
# CHINESE_FEW_SHOT: drawn exclusively from CMMD dev
# Both now include chain-of-thought reasoning step in example_answer()

INDIAN_FEW_SHOT = [
    # ── MISOGYNY (5) ──────────────────────────────────────────────────────
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/1110.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Two-panel Tamil meme. Top: Tamil text advising that when unsure what to cook, "
            "open a container for ideas. Bottom: Tamil comedian (Vadivelu) looking at a container "
            "labelled 'இல்லத்தரசிகள்' (Housewives). Hashtag #ரவை (Rava/Semolina) visible. "
            "The audience is explicitly addressed as housewives."
        ),
        "reasoning": (
            "Step 1 - Visual: The meme targets housewives as the audience, explicitly labelling "
            "women by their domestic role. Step 2 - Cultural: In Indian culture, women are often "
            "reduced to their role as homemakers. This meme reinforces that stereotype. "
            "Step 3 - Classification: Misogynistic because it reduces women's identity to domestic cooking."
        ),
        "explanation": (
            "The meme frames housewives as the audience for a cooking tip, explicitly labelling "
            "women as 'இல்லத்தரசிகள்' (Housewives) and reducing their identity to that of "
            "domestic cooks. It reinforces the gender-based assumption that cooking and "
            "domestic duties are a woman's primary identity and responsibility."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/1096.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Two-panel Tamil meme. Top panel: comedian actor (Vadivelu) labelled '*singles' "
            "with text 'Oru poi yavathu sol kannae.. Kanae' (At least tell one lie, dear). "
            "Bottom panel: group of men from a Tamil film labelled '*Love failures' "
            "with text 'Ava solrathu ellame poi than' (Everything she says is a lie). "
            "Contrasting expressions — longing vs. bitter."
        ),
        "reasoning": (
            "Step 1 - Visual: Two contrasting male perspectives on women are shown. "
            "Step 2 - Cultural: The meme frames women in romantic contexts as inherently deceptive, "
            "a common misogynistic trope in Indian pop culture. "
            "Step 3 - Classification: Misogynistic — stereotypes all women as liars."
        ),
        "explanation": (
            "The meme contrasts single men longing for female attention with 'love failure' "
            "men who claim women are congenital liars. By framing all women in romantic contexts "
            "as inherently deceptive, the meme reinforces a misogynistic stereotype that women "
            "manipulate and lie to men, targeting women's trustworthiness based solely on gender."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/1250.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Illustrated cartoon meme. Standing man gesturing in frustration toward a "
            "seated woman eating messily from a bowl on a sofa, dirty dishes piled on a side table. "
            "Tamil text at top: 'Love is just suffering… Marriage is suffering untold…' "
            "Way2news app watermark at bottom. Cartoon art style with exaggerated expressions."
        ),
        "reasoning": (
            "Step 1 - Visual: Wife shown as lazy/negligent, husband as exasperated victim. "
            "Step 2 - Cultural: In Indian cultural context, wives are expected to maintain domestic "
            "order. The meme implies women cause marital suffering through domestic negligence. "
            "Step 3 - Classification: Misogynistic — portrays wife as burden who ruins marriage."
        ),
        "explanation": (
            "The illustration frames the wife as lazy and negligent while the husband is exasperated. "
            "Paired with text equating marriage to 'untold suffering,' the meme implies women cause "
            "marital unhappiness through domestic negligence, reinforcing the misogynistic stereotype "
            "of wives as burdens who ruin domestic life after marriage."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/1440.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Real photograph of an Indian bride on her wedding day. She wears multiple "
            "heavy gold necklaces and bridal jewellery covering almost the entire neck and chest. "
            "Tamil text overlay: 'Due to lack of space to tie the mangalsutra (wedding necklace), "
            "the marriage is temporarily postponed.' No other visual elements."
        ),
        "reasoning": (
            "Step 1 - Visual: A real bride's photograph is used as comedic subject. "
            "Step 2 - Cultural: The joke targets the bride's body size implicitly, suggesting "
            "excessive jewellery (a proxy for body size) prevents marriage. Body shaming of women "
            "on their wedding day is a recognised form of misogyny in Indian culture. "
            "Step 3 - Classification: Misogynistic — body-shames a real woman."
        ),
        "explanation": (
            "The meme uses the bride's photograph to mock excessive jewellery as a barrier to "
            "marriage, implicitly targeting her body size. This is targeted body-shaming of a real "
            "woman on her wedding day, reinforcing the harmful idea that a woman's body is a valid "
            "subject of public ridicule."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/1234.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Four-panel comparison meme. Top-left: Tamil actor (Jayam Ravi) pointing. "
            "Top-right: Dora the Explorer cartoon. Bottom-left: Tamil comedian pointing. "
            "Bottom-right: Real woman in a saree walking on a road. Text repeated: "
            "'இவ தான் ஊர் சுத்தி' (She is the one who roams the neighbourhood)."
        ),
        "reasoning": (
            "Step 1 - Visual: A real woman is equated with a cartoon explorer as a 'roamer'. "
            "Step 2 - Cultural: 'ஊர் சுத்தி' carries sexually derogatory connotations for women "
            "in Tamil culture, implying promiscuity. The meme polices women's freedom of movement. "
            "Step 3 - Classification: Misogynistic — slut-shames a woman for walking in public."
        ),
        "explanation": (
            "'ஊர் சுத்தி' carries negative/sexually derogatory connotations when applied to women "
            "in Tamil, implying promiscuity. The meme slut-shames a real woman simply for being "
            "seen walking in public, reinforcing misogynistic policing of women's freedom of movement."
        ),
    },

    # ── NOT-MISOGYNY (5) ──────────────────────────────────────────────────
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/1006.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Four-panel 'Practical Exam's Be Like' meme. Top-left: relaxed external "
            "examiner with snacks. Top-right: confident student. Bottom-left: stern internal "
            "staff. Bottom-right: nervous student. Minion Memes Tamil watermark."
        ),
        "reasoning": (
            "Step 1 - Visual: Two types of examiners contrasted with student reactions. "
            "Step 2 - Cultural: The humour targets the exam system universally, not any gender. "
            "No women are specifically targeted or stereotyped. "
            "Step 3 - Classification: Not misogynistic — universal exam humour."
        ),
        "explanation": (
            "The meme humorously contrasts relaxed external examiners with feared internal staff. "
            "The humour targets the exam system universally. No gender-based stereotyping, "
            "objectification, or discrimination is present."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/360.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Facebook post. Text about trying to sleep / mosquito in ears. "
            "Below: three reaction shots of a young woman making animated expressions "
            "captioned 'Hey unnathan..unnathaannn' (mimicking mosquito buzz)."
        ),
        "reasoning": (
            "Step 1 - Visual: Woman's expressions used as comedic prop for mosquito character. "
            "Step 2 - Cultural: The humour is about a universal bedtime nuisance; the woman's "
            "expressions are used playfully, not to target or demean her as a woman. "
            "Step 3 - Classification: Not misogynistic — situational humour, no gender targeting."
        ),
        "explanation": (
            "The meme uses a woman's animated expressions to personify an annoying mosquito. "
            "The humour is situational and directed at a common everyday nuisance; the woman "
            "is used as a comedic prop for the mosquito, not targeted based on gender."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/656.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Dark-background text meme. Top text about deciding to study 'someday'. "
            "Movie still shows a man looking at a woman in traditional dress, labelled '*My laptop'. "
            "Vera Level Memes watermark."
        ),
        "reasoning": (
            "Step 1 - Visual: A woman from a movie clip is used as metaphor for a slow laptop. "
            "Step 2 - Cultural: The humour is about student procrastination and technology. "
            "The woman in the clip represents the laptop metaphorically — she is not being "
            "targeted or demeaned as a woman. "
            "Step 3 - Classification: Not misogynistic — tech/procrastination humour."
        ),
        "explanation": (
            "The meme uses a Tamil movie still to personify the student's slow laptop. "
            "The humour applies equally to all students regardless of gender; no gender "
            "stereotyping, objectification, or discrimination is present."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/214.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Facebook post. Text about frustration with Indian Cricket Team. "
            "Tamil film still showing two men in tense confrontation with subtitle "
            "'don't ask me about that team'. GOLDMINES entertainment watermark."
        ),
        "reasoning": (
            "Step 1 - Visual: Two men in a movie confrontation used to express sports frustration. "
            "Step 2 - Cultural: The humour is entirely about cricket culture and fan sentiment. "
            "No women are referenced or targeted. "
            "Step 3 - Classification: Not misogynistic — sports humour with no gender dimension."
        ),
        "explanation": (
            "The meme expresses comedic frustration at the Indian cricket team's performance. "
            "No women are referenced, targeted, or stereotyped. The humour is entirely about "
            "sports and has no gender-based dimension."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD/dev/787.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Multi-panel collage. Photographs of APJ Abdul Kalam, Virender Sehwag, "
            "Amitabh Bachchan, two South Indian actresses, Mahatma Gandhi. "
            "Text: 'Bold and Beautiful People are born in OCTOBER'. "
            "Bottom: Tamil movie still of young man boasting to friends."
        ),
        "reasoning": (
            "Step 1 - Visual: Positive collage of famous October-born people of both genders. "
            "Step 2 - Cultural: Birthday-month pride meme celebrating diverse public figures "
            "inclusively, both male and female. No gender targeting. "
            "Step 3 - Classification: Not misogynistic — inclusive celebratory content."
        ),
        "explanation": (
            "The meme celebrates October-born people across genders, featuring both male and female "
            "public figures equally. The self-congratulatory humour is inclusive and contains no "
            "gender-based targeting, stereotyping, objectification, or discrimination."
        ),
    },
]


CHINESE_FEW_SHOT = [
    # ── MISOGYNY (5) ──────────────────────────────────────────────────────
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/204.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Chinese ink-style illustrated meme. Two plump traditional female figures "
            "in red robes. One says 'EN EN' (agreeing); the other says '听我说' (Listen to me). "
            "Text: '孩子不听话是你不对 / 孩子学习成绩不好 / 还是你不对' "
            "(If child misbehaves it's your fault / If child's grades are poor / still your fault)."
        ),
        "reasoning": (
            "Step 1 - Visual: Mother figures blamed for all child outcomes, father absent from blame. "
            "Step 2 - Cultural: In Chinese family culture, mothers bear sole child-rearing responsibility. "
            "The meme reinforces this gender-based double standard. "
            "Step 3 - Classification: Misogynistic — perpetuates mother-blame stereotype."
        ),
        "explanation": (
            "The meme places sole blame on the mother for a child's misbehaviour and poor academic "
            "performance, with no mention of the father's responsibility. This perpetuates the "
            "stereotype that child-rearing failures fall exclusively on the mother."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/1303.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Minimalist text-card meme. Large title: '一定是女的' (Must be a woman). "
            "Three bullet examples: STEM girls learn worse than boys / "
            "Playing games badly must be a woman / Driving poorly must be a female driver."
        ),
        "reasoning": (
            "Step 1 - Visual: Three domains of incompetence attributed to being female. "
            "Step 2 - Cultural: The 'female driver' stereotype and STEM gender gap stereotypes "
            "are widely recognised misogynistic tropes in Chinese internet culture. "
            "Step 3 - Classification: Misogynistic — treats being female as explanation for incompetence."
        ),
        "explanation": (
            "The meme uses 'must be a woman' as the punchline for three separate domains of "
            "incompetence — academics, gaming, and driving. It explicitly treats being female "
            "as sufficient explanation for poor performance, perpetuating harmful gender stereotypes."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/932.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Hand-drawn illustration. Three figures at table: older girl (upset), "
            "adult mother (centre), younger boy (happily eating fries). "
            "Title: '只因为我是姐姐 / 我就要学会「谦让」' (Just because I'm the older sister "
            "/ I must learn to yield). Mother says: share fries with your brother."
        ),
        "reasoning": (
            "Step 1 - Visual: Older sister told to yield food to brother because she is female. "
            "Step 2 - Cultural: Son-preference in Chinese families is a documented form of "
            "institutional gender discrimination where daughters sacrifice for sons. "
            "Step 3 - Classification: Misogynistic — depicts systematic gender-based resource inequality."
        ),
        "explanation": (
            "The meme shows an older sister told to yield her food to a younger brother purely "
            "because she is the girl in the family. This depicts institutionalised gender "
            "discrimination where daughters are systematically expected to sacrifice for sons."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/1589.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Text-only meme. Cascading text: Don't study → fail exams → must get married "
            "→ give birth 3 times in 2 years → husband won't love you → mother-in-law abuses you "
            "→ wash clothes, sweep floors, cook and shop. '嫁人' (must get married) highlighted red."
        ),
        "reasoning": (
            "Step 1 - Visual: A bleak chain of consequences shown exclusively for girls. "
            "Step 2 - Cultural: The meme frames marriage as punishment and reduces a woman's future "
            "to domestic servitude and suffering, reflecting deeply patriarchal expectations in "
            "Chinese society about girls' worth being tied to education and marriage. "
            "Step 3 - Classification: Misogynistic — reduces women's fate to domestic oppression."
        ),
        "explanation": (
            "The meme presents a grim chain of consequences exclusively for girls: failure in "
            "education leads to forced marriage, domestic servitude, and abuse. By framing "
            "marriage as punishment and reducing women's future to household labour, the meme "
            "reinforces deeply misogynistic expectations about girls' worth."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/1656.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Text-card meme. Title: '外貌攻击' (Appearance Attack). Four quoted examples: "
            "'Long hair, short on knowledge' / 'How did you get so dark' / "
            "'Legs so thick still wearing a short skirt' / 'Looking like this, capability is useless'."
        ),
        "reasoning": (
            "Step 1 - Visual: Four appearance-based attacks catalogued, all directed at women. "
            "Step 2 - Cultural: The 'long hair, short on knowledge' proverb is a sexist Chinese "
            "idiom dismissing women's intelligence. Skin tone and body shaming are common forms "
            "of misogyny in Chinese internet culture. "
            "Step 3 - Classification: Misogynistic — body-shaming and appearance-based dismissal of women."
        ),
        "explanation": (
            "The meme catalogues appearance-based attacks at women: intelligence dismissed via "
            "the sexist proverb 'long hair, short on knowledge,' skin tone shaming, body shaming "
            "about leg size and clothing, and the claim that a woman's competence is worthless "
            "if she is not physically attractive."
        ),
    },

    # ── NOT-MISOGYNY (5) ──────────────────────────────────────────────────
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/423.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Photo of two black-and-white cats with exaggerated expressions. "
            "One cat holds playing cards. Text: '像我的生活...' (Like my life...) / "
            "'总是差一点就顺呐' (Always just a little short of going smoothly)."
        ),
        "reasoning": (
            "Step 1 - Visual: Funny cat expressions expressing universal life frustration. "
            "Step 2 - Cultural: The humour is relatable to everyone regardless of gender. "
            "No women are targeted or stereotyped. "
            "Step 3 - Classification: Not misogynistic — universal relatable humour."
        ),
        "explanation": (
            "The meme uses funny cat expressions to express universal frustration about life "
            "never going quite right. No gender targeting, no stereotyping, and no misogynistic "
            "content of any kind is present."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/1211.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Two-panel cat interview meme. Top: cat at desk saying don't need high-paying "
            "job, just find one you love. Bottom: same cat sheepishly admitting it likes high-paying jobs."
        ),
        "reasoning": (
            "Step 1 - Visual: Self-contradictory humour about job-hunting attitudes. "
            "Step 2 - Cultural: The joke targets universal workplace self-deception, applies "
            "equally to all genders. No women targeted. "
            "Step 3 - Classification: Not misogynistic — universal workplace humour."
        ),
        "explanation": (
            "The meme is a self-contradictory humour piece about job-hunting — preaching passion "
            "over salary then admitting wanting high salary. The joke targets universal workplace "
            "attitudes, applying equally to all genders. No women are targeted or demeaned."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/1403.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Office photo. Person at computer with printed meme taped to chair back. "
            "Meme jokes: 'National Day 7 days + Mid-Autumn 3 days / After make-up work = total 4 days off'. "
            "Comparing the Chinese 调休 system to a math trick."
        ),
        "reasoning": (
            "Step 1 - Visual: Workplace humour about the Chinese holiday scheduling system. "
            "Step 2 - Cultural: The 调休 (compensatory work-day) frustration is universal to "
            "all Chinese workers regardless of gender. "
            "Step 3 - Classification: Not misogynistic — workplace policy humour."
        ),
        "explanation": (
            "The meme jokes about the Chinese 调休 system converting long holidays into short ones. "
            "This is a universally shared workplace frustration. No misogynistic content present."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/1238.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Two-panel cartoon dog meme. Top: cute dog atop pile of papers, smiling. "
            "Text: '大难临头' (When disaster is upon you). Bottom: same dog folding papers into "
            "planes. Text: '先玩一会儿' (Play for a bit first). Bright cheerful style."
        ),
        "reasoning": (
            "Step 1 - Visual: Dog character procrastinating when facing overwhelming workload. "
            "Step 2 - Cultural: The procrastination joke is universally relatable, applies to "
            "everyone regardless of gender. "
            "Step 3 - Classification: Not misogynistic — universal procrastination humour."
        ),
        "explanation": (
            "The meme humorously captures the universal tendency to procrastinate when facing "
            "an overwhelming workload. The joke applies to everyone regardless of gender. "
            "No women are referenced, targeted, or stereotyped."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/CMMD/dev/544.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Two-panel Shiba Inu dog meme. Top: dog with tearful eyes, text: "
            "'以前：为什么不喜欢我' (Before: Why don't you like me). "
            "Bottom: dog with confident expression, text: '现在：没品味的东西' (Now: You have no taste)."
        ),
        "reasoning": (
            "Step 1 - Visual: Dog character expressing emotional arc from insecurity to confidence. "
            "Step 2 - Cultural: The rejection-to-confidence glow-up is a universal emotional "
            "experience with no gender-specific framing. "
            "Step 3 - Classification: Not misogynistic — universal emotional humour."
        ),
        "explanation": (
            "The meme expresses the relatable emotional arc from past insecurity about rejection "
            "to present self-confidence. The dog character is gender-neutral and the humour "
            "targets shared emotional experiences. No women are targeted or demeaned."
        ),
    },
]

MALAYALAM_FEW_SHOT = [
    # ── MISOGYNY (5) ──────────────────────────────────────────────────────
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/727.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Multi-panel meme titled '(June 19) A day in the life of an average Kajal "
            "devotee'. Top-left: actress in a casual floral dress, smiling. Top-middle: actress "
            "reclining in a swimsuit-style outfit, labelled 'noon'. Top-right: actress from "
            "behind in a low-back blouse and saree, labelled 'night'. Bottom: a man's face "
            "reacting with the caption 'Machane... athu poraaliya' (Bro... that's a warrior). "
            "Watermark: ACTRESS TROLLERS."
        ),
        "reasoning": (
            "Step 1 - Visual: A male fan's 'daily routine' is illustrated entirely through three "
            "different revealing photographs of an actress, framed as morning/noon/night viewing. "
            "Step 2 - Cultural: The meme reduces the actress to an object of male visual "
            "consumption across a day, a recognised misogynistic pattern in South Indian "
            "'actress-troll' pages. Step 3 - Classification: Misogynistic — objectifies a real "
            "woman's body as recurring visual content for male gratification."
        ),
        "explanation": (
            "The meme frames an entire day around consuming photographs of an actress in "
            "progressively more revealing outfits, with a male viewer's admiring reaction as the "
            "punchline. It reduces the actress to an object of visual consumption, typical of "
            "'actress troll' pages that objectify women's bodies for entertainment."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/580.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Six-panel collage of different actresses in casual wear, saree, and black "
            "top. Caption: 'Even without a smoking-hot body to show off, just the face is enough "
            "to turn us on — that is the achievement of many women.' Watermark: ട്രോൾ നടിമാർ "
            "(Troll Actresses)."
        ),
        "reasoning": (
            "Step 1 - Visual: Multiple actresses' faces and bodies are collaged and rated for "
            "their ability to arouse the viewer. Step 2 - Cultural: The meme explicitly frames a "
            "woman's 'achievement' as her capacity to sexually attract men, reducing her worth to "
            "appearance — a widely recognised misogynistic framing in South Indian meme culture. "
            "Step 3 - Classification: Misogynistic — defines women's value solely through their "
            "sexual appeal to men."
        ),
        "explanation": (
            "The meme collages six actresses and declares that a woman's real 'achievement' is "
            "her ability to sexually attract men even without a conventionally desirable body. "
            "This reduces women's worth entirely to their appeal to the male gaze, a clear "
            "misogynistic framing."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/43.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Three-panel dialogue meme between a man and his girlfriend at 3 AM. He "
            "interrogates her about whether she bathed and which soap she used. She answers "
            "'Chandrika'. He mocks her, calling her 'pazhutha Chandrika' (a stale/old Chandrika) "
            "and dismissively tells her to go back to sleep. Watermark: TMM / machante_troll."
        ),
        "reasoning": (
            "Step 1 - Visual: A man subjects his girlfriend to a late-night interrogation about "
            "her personal hygiene, then insults her using a soap-brand pun implying she is "
            "'old/stale'. Step 2 - Cultural: The joke frames the girlfriend as an object to be "
            "inspected and ridiculed at the man's whim, a common belittling dynamic toward women "
            "in Malayalam troll pages. Step 3 - Classification: Misogynistic — mocks and belittles "
            "a woman through a controlling, demeaning exchange."
        ),
        "explanation": (
            "The meme shows a man interrogating his girlfriend about her bathing habits late at "
            "night and then insulting her as 'stale' via a soap-brand pun. The joke's punchline "
            "is entirely at the woman's expense, reinforcing a demeaning, controlling dynamic "
            "toward her."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/546.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Two-panel 'Naughtiness reloaded' meme. Top: husband's face asking "
            "suspiciously, 'Girl, what are you doing alone in the room?', captioned "
            "'#a blue-film scene'. Bottom: wife peeking through a door gap saying, 'I'm just "
            "changing clothes, dear... I'll come now.' Watermark: MNT."
        ),
        "reasoning": (
            "Step 1 - Visual: A wife changing clothes in private is voyeuristically framed as "
            "suspicious/sexual, explicitly tagged as a 'blue-film' scene. Step 2 - Cultural: The "
            "meme sexualises a mundane private moment of a wife, reducing her to a voyeuristic "
            "spectacle for the viewer's titillation, consistent with 'naughty troll' pages that "
            "objectify women in domestic settings. Step 3 - Classification: Misogynistic — "
            "sexualises and objectifies a wife's private moment for the viewer's gratification."
        ),
        "explanation": (
            "The meme frames a wife changing clothes in her own room as a titillating "
            "'blue-film' moment, voyeuristically sexualising a private, mundane act. This "
            "objectifies the wife for the viewer's gratification rather than treating her as a "
            "person."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/575.jpg",
        "label":       1,
        "classification": "misogyny",
        "scene_graph": (
            "Scene: Four-panel collage of Malayalam actresses in sarees and traditional wear. "
            "Title: 'Malayalam cinema's menothikal (voluptuous women)' with punchline "
            "'അഥവാ മുലച്ചികൾ' (or, 'the busty ones')."
        ),
        "reasoning": (
            "Step 1 - Visual: Four actresses are labelled collectively by a vulgar term referring "
            "to breast size. Step 2 - Cultural: 'മുലച്ചികൾ' is a crude, dehumanising slur reducing "
            "women to a single body part; using it to caption respected actresses is a direct "
            "form of body-based objectification. Step 3 - Classification: Misogynistic — uses a "
            "vulgar slur to reduce women to a body part."
        ),
        "explanation": (
            "The meme labels a group of actresses with 'മുലച്ചികൾ', a crude slur that reduces "
            "them to a single body part. This directly objectifies and dehumanises the women "
            "pictured, regardless of the casual, jokey framing."
        ),
    },

    # ── NOT-MISOGYNY (5) ──────────────────────────────────────────────────
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/139.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Photo-edited meme of a woman standing waist-deep in a river holding a large "
            "blue 'Better luck next time' sign. Caption: 'When you throw the iron axe into the "
            "river hoping to get the golden axe.'"
        ),
        "reasoning": (
            "Step 1 - Visual: A woman is used as the visual stand-in for the well-known 'golden "
            "axe' honesty fable, holding a game-show-style 'better luck next time' placard. "
            "Step 2 - Cultural: The joke is about greed and the classic moral fable, not about "
            "the woman herself; her gender is incidental to the humour. "
            "Step 3 - Classification: Not misogynistic — universal moral-fable humour with no "
            "gender targeting."
        ),
        "explanation": (
            "The meme references the classic 'golden axe' fable about greed and honesty, using a "
            "woman merely as the visual prop delivering the 'better luck next time' punchline. "
            "The humour targets the fable's moral, not the woman's gender."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/799.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: News-style collage. Top: headline 'Kerala's second Vande Bharat flagged off'. "
            "Middle-left: a politician holding a national flag. Middle-right: the Vande Bharat "
            "train at a station. Bottom: a political leader being celebrated by supporters. "
            "Watermark: Troll Malayalam."
        ),
        "reasoning": (
            "Step 1 - Visual: Political and infrastructure news content — a train launch and a "
            "party leader celebration. Step 2 - Cultural: The content is entirely about politics "
            "and public transport; no individual or group is targeted by gender. "
            "Step 3 - Classification: Not misogynistic — political/news content with no gender "
            "dimension."
        ),
        "explanation": (
            "The meme reports on a train launch and a political celebration. No women are "
            "referenced, targeted, or stereotyped; the content is purely political/news "
            "commentary."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/513.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Three-panel promo collage for the family film 'Rahel Makan Kora'. Top: two "
            "women greeting each other warmly on a street. Middle: a son affectionately holding "
            "his mother's face at a railway station. Bottom: family members embracing near a "
            "bus. Captions praise it as a 'family entertainer' film embraced by youth and "
            "families alike."
        ),
        "reasoning": (
            "Step 1 - Visual: Warm family reunion and affection scenes from a movie promo. "
            "Step 2 - Cultural: The meme is a straightforward film promotional post highlighting "
            "family bonding; women are shown positively and affectionately, not stereotyped or "
            "demeaned. Step 3 - Classification: Not misogynistic — positive family-film "
            "promotional content."
        ),
        "explanation": (
            "The meme promotes a family-entertainer film through warm scenes of mothers, sons, "
            "and family members reuniting affectionately. Women are portrayed positively within "
            "a family context, with no stereotyping, objectification, or discrimination."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/204.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Close-up of a man's face with a fishing hook in the foreground, smiling "
            "wistfully. Caption: 'Feeling pity for the fish caught by the hook, I release it "
            "back into the water', captioned '*Manjira*' (a nickname). Watermark: Malayalam "
            "Troll Masters."
        ),
        "reasoning": (
            "Step 1 - Visual: A man expresses sentimental pity for a caught fish and releases it. "
            "Step 2 - Cultural: The humour is about a fishing anecdote and mock sentimentality; "
            "no woman is depicted, targeted, or referenced in a gendered way. "
            "Step 3 - Classification: Not misogynistic — gender-neutral fishing/sentimentality "
            "humour."
        ),
        "explanation": (
            "The meme is a light, self-deprecating joke about a man sparing a fish out of "
            "exaggerated pity. No women are depicted or referenced, and there is no gender-based "
            "content of any kind."
        ),
    },
    {
        "image_path":  f"{BASE_DIR}/MDMD_Malayalam/dev/155.jpg",
        "label":       0,
        "classification": "not-misogyny",
        "scene_graph": (
            "Scene: Three-panel dialogue between a father and son. The father asks the son to "
            "attend a relative's daughter's wedding since he is busy; the son negotiates back and "
            "forth about who should go, finally settling it based on whether the wedding serves "
            "a vegetarian feast or biryani. Watermark: Malayalam Trolls."
        ),
        "reasoning": (
            "Step 1 - Visual: A humorous father-son negotiation about attending a wedding, "
            "resolved by food preference. Step 2 - Cultural: The joke is about family obligation "
            "and food preference between two men; no woman is stereotyped, objectified, or "
            "targeted. Step 3 - Classification: Not misogynistic — father-son food/obligation "
            "humour."
        ),
        "explanation": (
            "The meme humorously depicts a father and son negotiating who must attend a wedding, "
            "ultimately deciding based on the menu. The joke is entirely between the two men over "
            "food preference and family obligation, with no gender-based targeting."
        ),
    },
]

ZERO_SHOT = []

# ── Partition-to-few-shot mapping ─────────────────────────────────────────────
PARTITION_FEW_SHOT = {
    "mami_indian":   INDIAN_FEW_SHOT,
    "mami_chinese":  CHINESE_FEW_SHOT,
    "mdmd_original": INDIAN_FEW_SHOT,
    "mdmd_original_malayalam": ZERO_SHOT,
    "mdmd_original_malayalam_few_shot_tamil": INDIAN_FEW_SHOT,
    "mdmd_original_malayalam_few_shot": MALAYALAM_FEW_SHOT,
    "mdmd_irish":    INDIAN_FEW_SHOT,
    "mdmd_chinese":  INDIAN_FEW_SHOT,
    "cmmd_original": CHINESE_FEW_SHOT,
    "cmmd_irish":    CHINESE_FEW_SHOT,
    "cmmd_indian":   CHINESE_FEW_SHOT,
}

print(f"Indian few-shot : {len(INDIAN_FEW_SHOT)} examples "
      f"({sum(1 for e in INDIAN_FEW_SHOT if e['label']==1)}M / "
      f"{sum(1 for e in INDIAN_FEW_SHOT if e['label']==0)}NM)")
print(f"Chinese few-shot: {len(CHINESE_FEW_SHOT)} examples "
      f"({sum(1 for e in CHINESE_FEW_SHOT if e['label']==1)}M / "
      f"{sum(1 for e in CHINESE_FEW_SHOT if e['label']==0)}NM)")
for partition, examples in PARTITION_FEW_SHOT.items():
    print(f"  {partition:20s} -> {len(examples)} examples")


In [ ]:
# ── Partition config ──────────────────────────────────────────────────────────
# is_cross_cultural=True  -> cross-cultural prompt + cultural marker injection
# is_cross_cultural=False -> native prompt, no markers (annotators share the
#                            meme's own culture, so no lens bridging is needed)

ALL_PARTITIONS = {

    # ── IN-CULTURE (native annotations, dev split) ───────────────────────────
    "mdmd_original": {
        "image_dir":         MDMD_DIR,
        "csv":               f"{MDMD_DIR}/test_mdmd_with_labels.csv",
        "country":           "India",
        "language":          "Tamil",
        "label_col":         "original_labels",
        "description":       "MDMD — native Indian perception (dev)",
        "is_cross_cultural": False,
    },
    "mdmd_original_malayalam": {
        "image_dir":         MDMD_MALAYALAM_DIR,
        "csv":               f"{MDMD_MALAYALAM_DIR}/test_malayalam_with_labels.csv",
        "country":           "India",
        "language":          "Malayalam",
        "label_col":         "original_labels",
        "description":       "MDMD — native Indian perception (dev)",
        "is_cross_cultural": False,
    },
    "mdmd_original_malayalam_few_shot_tamil": {
        "image_dir":         MDMD_MALAYALAM_DIR,
        "csv":               f"{MDMD_MALAYALAM_DIR}/test_malayalam_with_labels.csv",
        "country":           "India",
        "language":          "Malayalam",
        "label_col":         "original_labels",
        "description":       "MDMD Malayalam — few-shot, Tamil (MDMD) exemplars",
        "is_cross_cultural": False,
    },
    "mdmd_original_malayalam_few_shot": {
        "image_dir":         MDMD_MALAYALAM_DIR,
        "csv":               f"{MDMD_MALAYALAM_DIR}/test_malayalam_with_labels.csv",
        "country":           "India",
        "language":          "Malayalam",
        "label_col":         "original_labels",
        "description":       "MDMD — native Indian perception (dev)",
        "is_cross_cultural": False,
    },
    "cmmd_original": {
        "image_dir":         CMMD_DIR,
        "csv":               f"{CMMD_DIR}/test_cmmd_with_labels.csv",
        "country":           "China",
        "language":          "Chinese",
        "label_col":         "original_labels",
        "description":       "CMMD — native Chinese perception (dev)",
        "is_cross_cultural": False,
    },

    # ── CROSS-CULTURAL (MAMI test, no native label) ──────────────────────────
    "mami_indian": {
        "image_dir":         MAMI_DIR,
        "csv":               f"{MAMI_DIR}/test_with_labels_mami.csv",
        "country":           "India",
        "language":          "English",
        "label_col":         "indian_labels",
        "description":       "MAMI — Indian perception (test)",
        "is_cross_cultural": True,
    },
    "mami_chinese": {
        "image_dir":         MAMI_DIR,
        "csv":               f"{MAMI_DIR}/test_with_labels_mami.csv",
        "country":           "China",
        "language":          "English",
        "label_col":         "chinese_labels",
        "description":       "MAMI — Chinese perception (test)",
        "is_cross_cultural": True,
    },
}

PARTITIONS = {k: ALL_PARTITIONS[k] for k in RUN_PARTITIONS if k in ALL_PARTITIONS}

missing = [k for k in RUN_PARTITIONS if k not in ALL_PARTITIONS]
if missing:
    print(f"  [WARN] Unknown partition(s) ignored: {missing}")

print(f"Running {len(PARTITIONS)} partition(s): {list(PARTITIONS.keys())}")
for k, v in PARTITIONS.items():
    kind = "cross-cultural" if v["is_cross_cultural"] else "in-culture"
    print(f"  {k:16s} lens={v['country']:8s} label={v['label_col']:18s} [{kind}]")


In [ ]:
# ── Prompts and utilities ─────────────────────────────────────────────────────
import json, base64, glob, csv as csv_module, pandas as pd, os
from tqdm import tqdm
from PIL import Image

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Native partition prompt (MDMD/CMMD) ──────────────────────────────────────
NATIVE_PROMPT_TEMPLATE = """
You are a highly advanced misogyny detector, designed to analyse visual scenes
and evaluate memes for misogynistic content by examining both visual and
textual components, interpreted within the cultural, social, and linguistic
context of {country}.

You will be provided with a {language} meme image.

Step-by-Step Instructions:
1. Generate a scene graph by identifying all entities, objects, attributes,
   and relationships in the scene. Include characters, expressions, text
   overlays, symbols, layout, and background elements.
2. Use the scene graph from step 1 to analyse the visual elements and evaluate
   them against the socio-cultural context of {country}.

Classification Criteria:
- Classify as "misogyny" if the content directly or indirectly targets women
  or girls through: stereotyping, hatred, discrimination, sexualization,
  objectification, gender-based violence, insults, victim-blaming, mockery,
  or trivialization — even if conveyed through sarcasm or humor.
- Classify as "not-misogyny" if none of the above criteria apply.

Rules:
- Be objective, culturally aware, and precise.
- Do not treat humor or irony as neutral if it carries misogynistic meaning.
{ocr_section}
Respond ONLY in this JSON format (no markdown, no extra text):
{{
  "label": 1,
  "classification": "misogyny",
  "explanation": "Brief explanation referencing image and text with cultural context."
}}
label: 1 = misogyny, 0 = not-misogyny"""

# ── Cross-cultural (MAMI) prompt — IMPROVED with cultural markers ─────────────
# Changes from v1:
#   - Explicit bridging: evaluate Western meme through Indian/Chinese cultural lens
#   - Added culture-specific misogyny markers for both India and China
#   - 3-step evaluation explicitly named

INDIAN_CULTURAL_MARKERS = """
IMPORTANT — Indian annotators apply a NARROW, CONSERVATIVE standard.
When in doubt, answer not-misogyny.

Indian annotators DO label as misogyny:
- Explicit sexual objectification of women's bodies (breasts, buttocks, cleavage, nudity)
- Direct gendered slurs aimed at a woman
- Reduction of women to domestic servants (kitchen, sandwich, dishwasher tropes)
- Direct claims that women are incompetent (driving, logic, intelligence)

Indian annotators do NOT usually label as misogyny:
- Criticism or mockery of feminism or feminists as a movement
- General appearance or body-weight jokes without sexual framing
- Dark humour referencing sexual violence that is not aimed at a specific woman
- Relationship, wife, or marriage complaints framed as mutual banter
- Content that merely depicts a woman without demeaning her

Apply a high bar: flag misogyny only when the content explicitly sexualises,
degrades, or demeans women. Otherwise answer not-misogyny.
"""

CHINESE_CULTURAL_MARKERS = """
IMPORTANT — Chinese annotators apply a BROAD, INCLUSIVE standard.
About 52% of memes are labelled misogynistic, closely matching a general
Western misogyny judgement. When in doubt, lean toward misogyny.

Chinese annotators DO label as misogyny:
- Sexual objectification of women's bodies in any form
- Gendered slurs of any severity
- Anti-feminist content, including mockery of feminists or feminism
- Appearance-based ridicule: body weight, age, attractiveness
- Domestic role stereotypes (kitchen, cooking, cleaning tropes)
- Dismissal of women's competence, intelligence, or driving ability
- Jokes referencing sexual violence, harassment, or coercion
- Portraying women as materialistic, manipulative, or emotionally unstable
- Negative wife, girlfriend, or marriage stereotypes

Apply a low bar: if content demeans, sexualises, stereotypes, or diminishes
women in any way — including through humour, irony, or casual observation —
label it misogyny.
"""

CROSS_CULTURAL_PROMPT_TEMPLATE = """
You are a highly advanced misogyny detector evaluating memes from a cross-cultural perspective.

You will be provided with a {language} meme image from a Western/English context.
You must evaluate this meme from the perspective of an annotator with {country} cultural
values and sensitivities. Apply {country} cultural norms to assess whether this content
would be perceived as misogynistic by someone with an {country} cultural background,
even though the meme originates from a Western context.

{cultural_markers}

Step-by-Step Instructions:
1. Generate a scene graph by identifying all entities, objects, attributes,
   and relationships in the scene. Include characters, expressions, text
   overlays, symbols, layout, and background elements.
2. Use the scene graph from step 1 to analyse the visual and textual content.
3. Evaluate whether an annotator from {country} would consider this content
   misogynistic based on {country} cultural values — apply {country} standards,
   not Western standards alone.

Classification Criteria:
- Classify as "misogyny" if the content directly or indirectly targets women
  or girls through: stereotyping, hatred, discrimination, sexualization,
  objectification, gender-based violence, insults, victim-blaming, mockery,
  or trivialization — even if conveyed through sarcasm or humor.
- Classify as "not-misogyny" if none of the above criteria apply.

Rules:
- Be objective, culturally aware, and precise.
- Apply {country} cultural sensitivity, not Western standards alone.
- Do not treat humor or irony as neutral if it carries misogynistic meaning.
{ocr_section}
Respond ONLY in this JSON format (no markdown, no extra text):
{{
  "label": 1,
  "classification": "misogyny",
  "explanation": "Brief explanation referencing image and text with {country} cultural context."
}}
label: 1 = misogyny, 0 = not-misogyny"""

OCR_SECTION      = "\nOCR Transcription of meme text:\n\"\"\"\n{transcription}\n\"\"\"\n"
FALLBACK_PROMPT  = """Look at this meme image carefully.
Does it contain misogynistic content that targets, demeans, stereotypes,
or discriminates against women or girls?

Reply with ONLY a single digit: 1 for misogyny, 0 for not-misogyny.
No explanation needed. Just the digit."""

EXAMPLE_QUESTION = "Does this meme contain misogynistic content? Respond in JSON format."

CULTURAL_MARKERS_MAP = {
    "India":   INDIAN_CULTURAL_MARKERS,
    "China":   CHINESE_CULTURAL_MARKERS,
}


def load_pil(image_path: str):
    return Image.open(image_path).convert("RGB")


def build_prompt(language: str, country: str,
                 transcription: str = None,
                 is_cross_cultural: bool = False) -> str:
    ocr = OCR_SECTION.format(transcription=transcription) if transcription else ""
    if is_cross_cultural:
        markers = CULTURAL_MARKERS_MAP.get(country, "")
        return CROSS_CULTURAL_PROMPT_TEMPLATE.format(
            language=language, country=country,
            cultural_markers=markers, ocr_section=ocr)
    return NATIVE_PROMPT_TEMPLATE.format(
        language=language, country=country, ocr_section=ocr)


def example_answer(ex: dict) -> str:
    """
    Chain-of-thought few-shot answer.
    Includes scene_graph + explicit reasoning steps before the JSON.
    This shows the model HOW to reason, not just what the answer is.
    """
    reasoning = (
        f"Step 1 - Scene graph: {ex.get('scene_graph', '')}\n"
        f"Step 2 - Reasoning: {ex.get('reasoning', ex.get('explanation', ''))}\n"
        f"Step 3 - Classification:\n"
    )
    answer = json.dumps({
        "label":          ex["label"],
        "classification": ex["classification"],
        "explanation":    ex["explanation"],
    }, ensure_ascii=False)
    return reasoning + answer


def parse_response(result: str, image_path: str) -> tuple:
    if not result:
        return "not-misogyny", 0, "No response."
    try:
        text = result
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].strip()
        else:
            s = text.find("{"); e = text.rfind("}") + 1
            if s >= 0 and e > s:
                text = text[s:e]
        parsed = json.loads(text)
        label  = int(parsed.get("label", 0))
        return ("misogyny" if label == 1 else "not-misogyny",
                label, parsed.get("explanation", ""))
    except Exception as err:
        print(f"  [WARN] JSON parse failed for {os.path.basename(image_path)}: {err}")
        is_m = (result is not None
                and "misogyny" in result.lower()
                and "not-misogyny" not in result.lower()
                and "not misogyny" not in result.lower())
        return ("misogyny" if is_m else "not-misogyny",
                1 if is_m else 0, result or "Parse failed.")


print("Prompts and utilities ready.")
print(f"  Cross-cultural prompt: ACTIVE for MAMI partitions")
print(f"  Cultural markers: Indian + Chinese loaded")
print(f"  Few-shot format: Chain-of-thought (scene_graph + reasoning + JSON)")


In [ ]:
# ── Backend functions ─────────────────────────────────────────────────────────
import torch, gc
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from transformers import BitsAndBytesConfig

_hf_model     = None
_hf_processor = None
_hf_device    = "cuda" if torch.cuda.is_available() else "cpu"


def get_attn_implementation():
    """Select best attention implementation for current GPU."""
    if not torch.cuda.is_available():
        return "eager"
    # Try flash_attention_2, fall back to sdpa if not installed
    try:
        import flash_attn
        major = torch.cuda.get_device_capability()[0]
        if major >= 8:
            print("  flash_attention_2 available")
            return "flash_attention_2"
    except ImportError:
        pass
    print("  Using sdpa (flash-attn not installed)")
    return "sdpa"


def get_quantization_config():
    """4-bit on T4/V100, full precision on A100+."""
    if not torch.cuda.is_available():
        return None
    major = torch.cuda.get_device_capability()[0]
    if major >= 8:
        print("  A100+ detected → full bfloat16 (no quantisation)")
        return None   # A100: no quantisation needed
    print("  T4/V100 detected → 4-bit NF4 quantisation")
    return BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_compute_dtype    = torch.bfloat16,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_type       = "nf4",
    )


def get_processor_config():
    """Higher resolution on A100, reduced on T4 (memory)."""
    if not torch.cuda.is_available():
        return 128 * 28 * 28, 512 * 28 * 28
    major = torch.cuda.get_device_capability()[0]
    if major >= 8:
        return 256 * 28 * 28, 1280 * 28 * 28   # A100: full resolution
    return 128 * 28 * 28, 512 * 28 * 28         # T4: reduced resolution


def load_hf_model(model_name: str):
    global _hf_model, _hf_processor
    if _hf_model is not None:
        del _hf_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"Loading {model_name} on {_hf_device}...")
    quant_cfg  = get_quantization_config()
    attn_impl  = get_attn_implementation()
    min_px, max_px = get_processor_config()

    kwargs = dict(
        device_map          = "auto",
        attn_implementation = attn_impl,
    )
    if quant_cfg:
        kwargs["quantization_config"] = quant_cfg
    else:
        kwargs["torch_dtype"] = torch.bfloat16

    _hf_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name, **kwargs).eval()

    _hf_processor = AutoProcessor.from_pretrained(
        model_name, min_pixels=min_px, max_pixels=max_px)
    _hf_processor.tokenizer.padding_side = "left"

    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"Model ready. GPU free: {free_gb:.1f} GiB")


def _run_inference(messages: list, max_new_tokens: int = 512) -> str:
    from qwen_vl_utils import process_vision_info
    text             = _hf_processor.apply_chat_template(
                           messages, tokenize=False, add_generation_prompt=True)
    img_inp, vid_inp = process_vision_info(messages)
    inputs           = _hf_processor(
                           text=[text], images=img_inp, videos=vid_inp,
                           padding=True, return_tensors="pt").to(_hf_device)
    input_len        = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out_ids = _hf_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=False)
    return _hf_processor.decode(
        out_ids[0][input_len:], skip_special_tokens=True).strip()


def _build_messages(query_image_path: str, prompt: str,
                    few_shot_examples: list) -> list:
    """Build multi-turn few-shot messages list."""
    messages = []
    for ex in few_shot_examples:
        messages.append({"role": "user", "content": [
            {"type": "image", "image": load_pil(ex["image_path"])},
            {"type": "text",  "text": EXAMPLE_QUESTION},
        ]})
        messages.append({"role": "assistant", "content": example_answer(ex)})
    messages.append({"role": "user", "content": [
        {"type": "image", "image": load_pil(query_image_path)},
        {"type": "text",  "text": prompt},
    ]})
    return messages


def call_huggingface_single(query_image_path, prompt, few_shot_examples):
    return _run_inference(_build_messages(query_image_path, prompt, few_shot_examples))


def _try_parse_json(raw: str) -> bool:
    """Return True if raw string contains valid JSON."""
    try:
        text = raw
        if "```json" in text: text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:   text = text.split("```")[1].strip()
        else:
            s = text.find("{"); e = text.rfind("}") + 1
            if s >= 0 and e > s: text = text[s:e]
        json.loads(text)
        return True
    except Exception:
        return False


def _run_fallback(query_image_path: str) -> str:
    """Simplified binary fallback when JSON parse fails."""
    img_id = os.path.splitext(os.path.basename(query_image_path))[0]
    print(f"  [FALLBACK] Re-prompting {img_id} with simplified question")
    raw   = _run_inference([{"role": "user", "content": [
        {"type": "image", "image": load_pil(query_image_path)},
        {"type": "text",  "text": FALLBACK_PROMPT},
    ]}], max_new_tokens=16)
    digit = raw.strip()[:1]
    label = 1 if digit == "1" else 0
    return json.dumps({
        "label":          label,
        "classification": "misogyny" if label == 1 else "not-misogyny",
        "explanation":    f"Fallback classification. Raw: {raw[:100]}",
    })


def call_huggingface_with_fallback(query_image_path, prompt,
                                   few_shot_examples) -> tuple:
    raw = call_huggingface_single(query_image_path, prompt, few_shot_examples)
    if _try_parse_json(raw):
        return raw, False
    return _run_fallback(query_image_path), True


def call_huggingface_batch(query_image_paths, prompts,
                           few_shot_examples) -> list:
    """Batched inference. Now includes per-item fallback recovery."""
    from qwen_vl_utils import process_vision_info
    batch_texts, batch_images = [], []

    for qp, prompt in zip(query_image_paths, prompts):
        msgs   = _build_messages(qp, prompt, few_shot_examples)
        text   = _hf_processor.apply_chat_template(
                     msgs, tokenize=False, add_generation_prompt=True)
        imgs, _ = process_vision_info(msgs)
        batch_texts.append(text)
        batch_images.extend(imgs)

    inputs    = _hf_processor(text=batch_texts, images=batch_images,
                               padding=True, return_tensors="pt").to(_hf_device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out_ids = _hf_model.generate(**inputs, max_new_tokens=512, do_sample=False)

    raws = [_hf_processor.decode(out_ids[i][input_len:],
                                  skip_special_tokens=True).strip()
            for i in range(len(query_image_paths))]

    # Per-item fallback recovery for failed JSON (NEW in v2)
    results = []
    for qp, raw in zip(query_image_paths, raws):
        if not _try_parse_json(raw):
            raw = _run_fallback(qp)
        results.append(raw)
    return results


print("Backend functions ready.")


In [ ]:
# ── Classification functions ──────────────────────────────────────────────────

def norm_id(val):
    s = str(val).strip()
    return s[:-2] if s.endswith(".0") else s


def classify_image(image_path, model_name, country, language,
                   transcription=None, is_cross_cultural=False,
                   few_shot_examples=None):
    if few_shot_examples is None:
        raise ValueError(
            "few_shot_examples must be passed explicitly; use [] for zero-shot. "
            "Silent defaulting to INDIAN_FEW_SHOT previously caused a mis-run "
            "in which zero-shot and few-shot produced identical results.")
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    prompt   = build_prompt(language, country, transcription, is_cross_cultural)
    try:
        raw, used_fallback = call_huggingface_with_fallback(
            image_path, prompt, few_shot_examples)
    except Exception as e:
        return {"image_id": image_id, "label": -1,
                "classification": "error", "explanation": str(e),
                "full_response": str(e), "used_fallback": False}
    classification, label, explanation = parse_response(raw, image_path)
    return {"image_id": image_id, "label": label,
            "classification": classification, "explanation": explanation,
            "full_response": raw, "used_fallback": used_fallback}


def classify_batch(image_paths, model_name, country, language,
                   transcriptions, is_cross_cultural=False,
                   few_shot_examples=None):
    if few_shot_examples is None:
        raise ValueError(
            "few_shot_examples must be passed explicitly; use [] for zero-shot. "
            "Silent defaulting to INDIAN_FEW_SHOT previously caused a mis-run "
            "in which zero-shot and few-shot produced identical results.")
    prompts = [build_prompt(language, country,
                            transcriptions.get(os.path.splitext(os.path.basename(p))[0]),
                            is_cross_cultural)
               for p in image_paths]
    try:
        raws = call_huggingface_batch(image_paths, prompts, few_shot_examples)
    except Exception as e:
        return [{"image_id": os.path.splitext(os.path.basename(p))[0],
                 "label": -1, "classification": "error",
                 "explanation": str(e), "full_response": str(e),
                 "used_fallback": False}
                for p in image_paths]
    results = []
    for img_path, raw in zip(image_paths, raws):
        image_id = os.path.splitext(os.path.basename(img_path))[0]
        classification, label, explanation = parse_response(raw, img_path)
        results.append({"image_id": image_id, "label": label,
                        "classification": classification,
                        "explanation": explanation, "full_response": raw,
                        "used_fallback": False})
    return results


def batch_classify(partition_cfg, model_name, output_dir, partition_name):
    image_dir        = partition_cfg["image_dir"]
    csv_path         = partition_cfg.get("csv")
    country          = partition_cfg["country"]
    language         = partition_cfg["language"]
    desc             = partition_cfg["description"]
    partition_key    = partition_cfg["label_col"]
    is_cross_cultural = partition_cfg.get("is_cross_cultural", False)  # ← correct key

    # NOTE: partition_name is now passed in by the caller. It must NOT be
    # reverse-derived from (label_col, country): mdmd_original,
    # mdmd_original_malayalam and mdmd_original_malayalam_few_shot all share
    # ("original_labels", "India"), so next() returned mdmd_original for all
    # three and every Malayalam run silently used INDIAN_FEW_SHOT.
    if partition_name not in PARTITION_FEW_SHOT:
        raise KeyError(
            f"No few-shot pool registered for partition {partition_name!r}. "
            f"Add it to PARTITION_FEW_SHOT (use ZERO_SHOT for a zero-shot run).")
    few_shot_examples = PARTITION_FEW_SHOT[partition_name]

    transcriptions = {}
    if csv_path and os.path.exists(csv_path):
        df = pd.read_csv(csv_path, sep=None, engine="python")
        if "transcriptions" in df.columns and "image_id" in df.columns:
            transcriptions = {norm_id(k): str(v)
                              for k, v in zip(df["image_id"], df["transcriptions"])}
            print(f"  Loaded {len(transcriptions)} transcriptions")

    # Exclude exemplars by FULL PATH, and only those actually in use.
    # The previous version matched on bare filename ID across the union of the
    # Indian and Chinese pools, which wrongly dropped unrelated test memes that
    # happened to share an integer ID (Tamil lost 204/544/1238 -> n=353 instead
    # of 356; Malayalam lost 214/544 -> n=198 instead of 200). Exemplars live in
    # dev directories, so full-path matching excludes nothing from a test run.
    few_shot_paths = {os.path.abspath(ex["image_path"]) for ex in few_shot_examples}
    image_files = sorted([
        f for ext in ["*.jpg", "*.jpeg", "*.png"]
        for f in glob.glob(os.path.join(image_dir, ext))
        if os.path.abspath(f) not in few_shot_paths
    ])

    if DEBUG_LIMIT:
        image_files = image_files[:DEBUG_LIMIT]
        print(f"  [DEBUG] Limited to {DEBUG_LIMIT} images")

    if not image_files:
        print(f"  [WARN] No images found in {image_dir} — skipping.")
        return []

    print(f"\n{'='*60}")
    print(f"  Partition   : {desc}")
    print(f"  Model       : {model_name}")
    print(f"  Country     : {country}  |  Language: {language}")
    print(f"  Cross-cult. : {is_cross_cultural}")
    _mode = "ZERO-SHOT" if not few_shot_examples else f"FEW-SHOT ({len(few_shot_examples)} exemplars)"
    print(f"  Partition   : {partition_name}")
    print(f"  Prompting   : {_mode}")
    print(f"  Images      : {len(image_files)} (excluded {len(few_shot_paths)} exemplar path(s))")
    print(f"  Batch size  : {BATCH_SIZE}")
    print(f"{'='*60}")

    os.makedirs(output_dir, exist_ok=True)
    safe_model = model_name.replace("/", "_").replace(":", "-")
    out_base   = os.path.join(output_dir, f"{safe_model}__{partition_key}")
    json_out   = f"{out_base}.json"
    submit_out = f"{out_base}_submission.csv"

    results, fallback_count = [], 0

    for batch_start in tqdm(range(0, len(image_files), BATCH_SIZE),
                            desc=f"{model_name.split('/')[-1][:20]} | {country}",
                            unit="batch", colour="green"):
        batch_paths = image_files[batch_start:batch_start + BATCH_SIZE]
        if BATCH_SIZE == 1:
            result = classify_image(
                image_path        = batch_paths[0],
                model_name        = model_name,
                country           = country,
                language          = language,
                transcription     = transcriptions.get(
                    os.path.splitext(os.path.basename(batch_paths[0]))[0]),
                is_cross_cultural = is_cross_cultural,
                few_shot_examples = few_shot_examples,
            )
            if result.get("used_fallback"): fallback_count += 1
            results.append(result)
        else:
            batch_results = classify_batch(
                image_paths       = batch_paths,
                model_name        = model_name,
                country           = country,
                language          = language,
                transcriptions    = transcriptions,
                is_cross_cultural = is_cross_cultural,
                few_shot_examples = few_shot_examples,
            )
            fallback_count += sum(1 for r in batch_results if r.get("used_fallback"))
            results.extend(batch_results)

        import torch as _torch
        _torch.cuda.empty_cache()

    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    with open(submit_out, "w", newline="") as f:
        writer = csv_module.writer(f)
        writer.writerow(["image_id", "label"])
        for r in results:
            writer.writerow([r["image_id"], r["label"]])

    total    = len(results)
    misogyny = sum(1 for r in results if r["label"] == 1)
    not_m    = sum(1 for r in results if r["label"] == 0)
    errors   = sum(1 for r in results if r["label"] == -1)
    print(f"\n  Results -> {json_out}")
    print(f"  Submit  -> {submit_out}")
    print(f"  Summary : total={total}  misogyny={misogyny}  "
          f"not-misogyny={not_m}  errors={errors}  fallbacks={fallback_count}")
    return results


print("All functions ready.")


In [ ]:
# ── Run ──────────────────────────────────────────────────────────────────────

prev_model = None

for model_name in SELECTED_MODELS:
    if model_name != prev_model:
        load_hf_model(model_name)
        prev_model = model_name
    for partition_name, partition_cfg in PARTITIONS.items():
        out_dir = os.path.join(OUTPUT_DIR, partition_name)
        batch_classify(partition_cfg=partition_cfg,
                       model_name=model_name, output_dir=out_dir,
                       partition_name=partition_name)

print("\nAll runs complete.")


In [ ]:
# ── Score against ground truth ───────────────────────────────────────────────

def norm_label(val) -> int:
    if isinstance(val, (int, float)): return int(val)
    s = str(val).strip().lower()
    if s in ("1", "misogyny"):        return 1
    if s in ("0", "not-misogyny", "not misogyny"): return 0
    raise ValueError(f"Unrecognised label value: {val!r}")


def score_partition(results_json_path: str, label_csv_path: str,
                    label_col: str) -> dict:
    from sklearn.metrics import f1_score, accuracy_score
    from collections import Counter

    if not os.path.exists(results_json_path):
        print(f"  [SKIP] Not found: {results_json_path}")
        return {}

    with open(results_json_path) as f:
        results = json.load(f)
    df = pd.read_csv(label_csv_path, sep=None, engine="python")
    df["image_id"] = df["image_id"].apply(norm_id)
    pred_map = {r["image_id"]: r["label"]
                for r in results if str(r["label"]) not in ("-1", "error")}
    y_true, y_pred = [], []
    for _, row in df.iterrows():
        iid = norm_id(row["image_id"])
        if iid in pred_map and pd.notna(row.get(label_col)):
            y_true.append(norm_label(row[label_col]))
            y_pred.append(int(pred_map[iid]))
    if not y_true:
        print("  [WARN] No matching image IDs found.")
        return {}
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\n  Macro-F1 : {macro_f1:.4f}")
    print(f"  Accuracy : {accuracy:.4f}")
    print(f"  Samples  : {len(y_true)}")
    print(f"  Predicted   distribution: {dict(Counter(y_pred))}")
    print(f"  Groundtruth distribution: {dict(Counter(y_true))}")
    return {"macro_f1": macro_f1, "accuracy": accuracy, "n": len(y_true)}


# ── Score every partition that has results on disk ───────────────────────────
MODEL_TAG = SELECTED_MODELS[0].replace("/", "_")

SCORE_TARGETS = {
    "mdmd_original": (f"{MDMD_DIR}/test_mdmd_with_labels.csv", "original_labels"),
    "mdmd_original_malayalam": (f"{MDMD_MALAYALAM_DIR}/test_malayalam_with_labels.csv", "original_labels"),
    "mdmd_original_malayalam_few_shot_tamil": (f"{MDMD_MALAYALAM_DIR}/test_malayalam_with_labels.csv", "original_labels"),
    "mdmd_original_malayalam_few_shot": (f"{MDMD_MALAYALAM_DIR}/test_malayalam_with_labels.csv", "original_labels"),
    "cmmd_original": (f"{CMMD_DIR}/test_cmmd_with_labels.csv",   "original_labels"),
    "mami_indian":   (f"{MAMI_DIR}/test_with_labels_mami.csv",   "indian_labels"),
    "mami_chinese":  (f"{MAMI_DIR}/test_with_labels_mami.csv",   "chinese_labels"),
}

all_scores = {}
for partition, (csv_path, label_col) in SCORE_TARGETS.items():
    json_path = f"{OUTPUT_DIR}/{partition}/{MODEL_TAG}__{label_col}.json"
    print(f"\n{'='*58}")
    print(f"  {partition}  ({label_col})")
    print(f"{'='*58}")
    s = score_partition(json_path, csv_path, label_col)
    if s:
        all_scores[partition] = s

# ── Summary table ────────────────────────────────────────────────────────────
if all_scores:
    print(f"\n{'='*58}")
    print(f"  {'Partition':16s} {'Macro-F1':>10s} {'Accuracy':>10s} {'N':>7s}")
    print(f"  {'-'*54}")
    for p, s in all_scores.items():
        print(f"  {p:16s} {s['macro_f1']:10.4f} {s['accuracy']:10.4f} {s['n']:7d}")
    print(f"{'='*58}")


In [ ]:
# ── Download results ─────────────────────────────────────────────────────────
import shutil
from google.colab import files

zip_path = "/content/cc_mmd_results_v2"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(f"{zip_path}.zip")
print("Downloaded cc_mmd_results_v2.zip")
